<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week6_Day3_Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [3]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Remplacement du TODO par une phrase réelle
sample_sentence = "Hugging Face makes NLP accessible to everyone."
print(f"Sentence: {sample_sentence}")

Sentence: Hugging Face makes NLP accessible to everyone.


In [6]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=15,  # Ajusté pour la phrase d'exemple
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | hugging      | 17662
    2 | face         |  2227
    3 | makes        |  3084
    4 | nl           | 17953
    5 | ##p          |  2361
    6 | accessible   |  7801
    7 | to           |  2000
    8 | everyone     |  3071
    9 | .            |  1012
   10 | [SEP]        |   102
   11 | [PAD]        |     0
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (10, '[SEP]'), (11, '[PAD]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]')]


### Exercise 1 reflection
- **[CLS] et [SEP]** : `[CLS]` (Classification) est ajouté au début pour capturer une représentation globale de la séquence, tandis que `[SEP]` (Separator) marque la fin d'une phrase ou la séparation entre deux phrases.
- **Attention Mask** : Le masque d'attention contient des `1` pour les jetons réels et des `0` pour le rembourrage (`[PAD]`). Cela indique au mécanisme d'auto-attention de BERT d'ignorer les positions `0` lors du calcul des scores, évitant ainsi que le bruit du padding n'influence les représentations.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [7]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This tutorial on BERT is incredibly helpful and easy to follow!"
prediction = sentiment_pipeline(sentence)
print(f"Sentence: {sentence}")
print(f"Prediction: {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence: This tutorial on BERT is incredibly helpful and easy to follow!
Prediction: [{'label': 'POSITIVE', 'score': 0.9995172023773193}]


### Exercise 2 reflection
- **Attente** : Le label 'POSITIVE' correspond parfaitement à la phrase enthousiaste fournie.
- **Confiance** : Le score (souvent > 0.99) indique que le modèle est très sûr de lui. Ce score représente la probabilité assignée à la classe après le passage dans la fonction softmax.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        return self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)

        score, index = torch.max(probs, dim=1)
        label = self.model.config.id2label[index.item()]
        return {"label": label, "probability": score.item()}

In [9]:
# Test de l'analyseur personnalisé
analyzer = BERTSentimentAnalyzer()
samples = [
    "I absolutely love how transformers work!",
    "I am very disappointed with the slow performance."
]
for text in samples:
    result = analyzer.predict(text)
    print(f"Text: {text}")
    print(f"Result: {result}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I absolutely love how transformers work!
Result: {'label': 'POSITIVE', 'probability': 0.9998301267623901}

Text: I am very disappointed with the slow performance.
Result: {'label': 'NEGATIVE', 'probability': 0.999808132648468}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [10]:
from transformers import pipeline
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = 0 if torch.cuda.is_available() else -1
        # Utilisation de la pipeline pour simplifier la gestion des offsets et du groupement d'entités
        self.ner_pipeline = pipeline(
            "ner",
            model=model_name,
            tokenizer=model_name,
            device=self.device,
            aggregation_strategy="simple" # Fusionne automatiquement les ##wordpieces
        )

    def recognize(self, text: str):
        return self.ner_pipeline(text)

In [11]:
ner = BERTNamedEntityRecognizer()
sample_text = "Hugging Face is a company based in New York City and Paris, founded by Clement Delangue."

entities = ner.recognize(sample_text)

print(f"Texte: {sample_text}\n")
print(f"{'Entity':<15} | {'Label':<10} | {'Score':<10}")
print("-" * 40)
for ent in entities:
    print(f"{ent['word']:<15} | {ent['entity_group']:<10} | {ent['score']:.4f}")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Texte: Hugging Face is a company based in New York City and Paris, founded by Clement Delangue.

Entity          | Label      | Score     
----------------------------------------
Hugging Face    | ORG        | 0.9524
New York City   | LOC        | 0.9996
Paris           | LOC        | 0.9996
Clement Delangue | PER        | 0.8582


## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encodeur (bidirectionnel) | Décodeur (unidirectionnel/autorégressif) |
| Primary purpose | Compréhension du langage (NLU) | Génération de texte (NLG) |
| Typical use cases | Classification, NER, Recherche sémantique | Chatbots, Rédaction, Résumé |
| Strengths | Contexte bidirectionnel complet | Capacités de génération créative et fluide |
| Weaknesses | Ne peut pas générer de longs textes | Moins efficace pour l'extraction de données structurées |

## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **Encodage** : BERT transforme les requêtes et les documents en vecteurs denses (embeddings) qui capturent la sémantique profonde plutôt que de simples mots-clés.
2. **Stockage et Recherche** : Ces vecteurs sont stockés dans une base de données vectorielle (comme Pinecone ou Milvus). On effectue une recherche de similarité cosinus pour trouver les documents les plus proches de la requête.
3. **Génération** : Les passages récupérés sont insérés dans le 'prompt' envoyé à un modèle comme GPT, lui fournissant ainsi un contexte externe et à jour pour générer sa réponse.
4. **Application** : Un système de support technique interne pour une entreprise, où BERT indexe toute la documentation technique complexe pour aider un LLM à répondre précisément aux employés.